# Debit/Credit Card Fraud Detection using Machine Learning
Dataset: Sparkov (Kaggle) - `fraudTrain.csv` + `fraudTest.csv`  
`is_fraud = 1` -> Fraud, `is_fraud = 0` -> Genuine

**Plan:** load -> EDA -> feature engineering -> encode/scale -> balance (train only) -> compare -> tune threshold -> test -> save

## 1. Install and imports

In [ ]:
# Colab / Jupyter: uncomment the next line once
# !pip install xgboost imbalanced-learn -q

import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_auc_score, confusion_matrix, precision_recall_curve)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
RANDOM_STATE = 42

## 2. Load the dataset

In [ ]:
# --- Option A: files on your computer / Google Drive ---
DATA_DIR = "data"            # folder that contains fraudTrain.csv and fraudTest.csv
TRAIN_FILE = f"{DATA_DIR}/fraudTrain.csv"
TEST_FILE  = f"{DATA_DIR}/fraudTest.csv"

# --- Option B: Google Colab with Kaggle (uncomment) ---
# import kagglehub
# DATA_DIR = kagglehub.dataset_download("kartik2112/fraud-detection")
# TRAIN_FILE = f"{DATA_DIR}/fraudTrain.csv"
# TEST_FILE  = f"{DATA_DIR}/fraudTest.csv"

NROWS = None   # for a quick trial use e.g. 200000 (first rows of each file). None = full data

train = pd.read_csv(TRAIN_FILE, index_col=0, nrows=NROWS)
test  = pd.read_csv(TEST_FILE,  index_col=0, nrows=NROWS)

print("Train shape:", train.shape)
print("Test  shape:", test.shape)
print("\nTrain fraud counts:\n", train["is_fraud"].value_counts())
print("Train fraud %:", round(train["is_fraud"].mean() * 100, 3))
print("Test  fraud %:", round(test["is_fraud"].mean() * 100, 3))
train.head()

## 3. Quick EDA (saved in `reports/`)

In [ ]:
eda = train.copy()
eda["trans_date_trans_time"] = pd.to_datetime(eda["trans_date_trans_time"])
eda["hour"] = eda["trans_date_trans_time"].dt.hour

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

eda["is_fraud"].value_counts().plot(kind="bar", ax=ax[0, 0], color=["#4C78A8", "#E45756"])
ax[0, 0].set_title("Class imbalance (0 = genuine, 1 = fraud)")

eda.groupby("hour")["is_fraud"].mean().mul(100).plot(ax=ax[0, 1], marker="o")
ax[0, 1].set_title("Fraud rate (%) by hour")

eda.groupby("category")["is_fraud"].mean().mul(100).sort_values().plot(kind="barh", ax=ax[1, 0])
ax[1, 0].set_title("Fraud rate (%) by category")

sns.boxplot(data=eda[eda["amt"] < eda["amt"].quantile(0.99)], x="is_fraud", y="amt", ax=ax[1, 1])
ax[1, 1].set_title("Amount: genuine vs fraud (below 99th percentile)")

plt.tight_layout()
plt.savefig("reports/eda_overview.png", dpi=150)
plt.show()
del eda

## 4. Feature engineering
Train and test are joined **only for building features**, so a card's first test transactions still see its history.  
Every history feature uses **past transactions only** (`shift`, `diff`, `closed="left"`), so there is no leakage.

In [ ]:
train["split"] = "train"
test["split"]  = "test"
full = pd.concat([train, test], ignore_index=True)

full["trans_date_trans_time"] = pd.to_datetime(full["trans_date_trans_time"])
full["dob"] = pd.to_datetime(full["dob"])
full = full.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)
t = full["trans_date_trans_time"]

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# ---------- Time ----------
full["hour"] = t.dt.hour
full["day_of_week"] = t.dt.dayofweek
full["month"] = t.dt.month
full["is_weekend"] = (full["day_of_week"] >= 5).astype(int)
full["is_night"] = full["hour"].isin([23, 0, 1, 2, 3, 4]).astype(int)

# ---------- Customer ----------
full["age"] = (t - full["dob"]).dt.days / 365.25

# ---------- Location ----------
full["distance_km"] = haversine(full["lat"], full["long"], full["merch_lat"], full["merch_long"])

g = full.groupby("cc_num")
full["prev_merch_lat"]  = g["merch_lat"].shift(1)
full["prev_merch_long"] = g["merch_long"].shift(1)
full["dist_from_prev"] = haversine(full["prev_merch_lat"], full["prev_merch_long"],
                                   full["merch_lat"], full["merch_long"])

# ---------- Gap, speed (impossible travel) ----------
full["time_since_last"] = g["trans_date_trans_time"].diff().dt.total_seconds()
hours = (full["time_since_last"] / 3600).clip(lower=1 / 60)       # at least 1 minute
full["speed_kmh"] = full["dist_from_prev"] / hours

# ---------- Amount behaviour ----------
full["prev_amt"] = g["amt"].shift(1)
full["amt_diff_prev"] = full["amt"] - full["prev_amt"]
full["card_avg_amt"] = g["amt"].transform(lambda s: s.shift(1).expanding().mean())
full["amt_ratio"] = full["amt"] / full["card_avg_amt"]

# ---------- Velocity: transactions in the last 1h / 24h (current row excluded) ----------
parts = []
for cc, grp in full.groupby("cc_num"):
    s = grp.set_index("trans_date_trans_time")["amt"]
    r1, r24 = s.rolling("1h", closed="left"), s.rolling("24h", closed="left")
    parts.append(pd.DataFrame({"txn_1h": r1.count().values,
                               "txn_24h": r24.count().values,
                               "amt_sum_24h": r24.sum().values}, index=grp.index))
roll = pd.concat(parts).loc[full.index]
full[["txn_1h", "txn_24h", "amt_sum_24h"]] = roll.fillna(0).values

# ---------- Clean first-transaction NaNs ----------
full = full.replace([np.inf, -np.inf], np.nan)
for c in ["time_since_last", "dist_from_prev", "speed_kmh", "amt_diff_prev"]:
    full[c] = full[c].fillna(0)
full["amt_ratio"] = full["amt_ratio"].fillna(1)

print("Features created. Shape:", full.shape)

## 5. Split back to train / test, encode and scale
Encoder and scaler are **fit on train only**.

In [ ]:
num_cols = ["amt", "hour", "day_of_week", "month", "is_weekend", "is_night",
            "age", "city_pop", "distance_km", "dist_from_prev", "speed_kmh",
            "time_since_last", "amt_diff_prev", "amt_ratio",
            "txn_1h", "txn_24h", "amt_sum_24h"]
cat_cols = ["category", "gender", "state"]
FEATURES = num_cols + cat_cols

train_df = full[full["split"] == "train"].sort_values("trans_date_trans_time").reset_index(drop=True)
test_df  = full[full["split"] == "test"].sort_values("trans_date_trans_time").reset_index(drop=True)

scaler = StandardScaler().fit(train_df[num_cols])
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(train_df[cat_cols])

def prepare(df):
    X = df[FEATURES].copy()
    X[num_cols] = scaler.transform(X[num_cols])
    X[cat_cols] = encoder.transform(X[cat_cols])
    return X

X_train_full, y_train_full = prepare(train_df), train_df["is_fraud"].values
X_test, y_test = prepare(test_df), test_df["is_fraud"].values

# Time-based validation: oldest 80% of train -> fit, newest 20% -> validation (used for threshold + method choice)
cut = int(len(train_df) * 0.8)
X_tr, y_tr = X_train_full.iloc[:cut], y_train_full[:cut]
X_val, y_val = X_train_full.iloc[cut:], y_train_full[cut:]

print("Fit part :", X_tr.shape, "fraud =", int(y_tr.sum()))
print("Validation:", X_val.shape, "fraud =", int(y_val.sum()))
print("Test     :", X_test.shape, "fraud =", int(y_test.sum()))

## 6. Balancing - increasing fraud examples
Balancing is applied to the **fit part only**. Validation and test stay original.

| Method | What it does |
|---|---|
| none | baseline |
| class_weight | gives fraud more weight, data unchanged |
| under | removes many genuine rows |
| under+smote | removes genuine rows, then **creates new synthetic fraud rows** (SMOTENC handles category columns) |

In [ ]:
cat_idx = [FEATURES.index(c) for c in cat_cols]

def balance(method, X, y):
    if method in ("none", "class_weight"):
        return X, y
    if method == "under":
        return RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_STATE).fit_resample(X, y)
    if method == "under+smote":
        Xu, yu = RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_STATE).fit_resample(X, y)
        return SMOTENC(categorical_features=cat_idx, sampling_strategy=0.5,
                       random_state=RANDOM_STATE).fit_resample(Xu, yu)
    raise ValueError(method)

def best_threshold(y_true, prob):
    prec, rec, thr = precision_recall_curve(y_true, prob)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return float(thr[np.argmax(f1)])

def scores(y_true, prob, thr):
    pred = (prob >= thr).astype(int)
    return {"precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "pr_auc": average_precision_score(y_true, prob),
            "roc_auc": roc_auc_score(y_true, prob)}

def make_xgb(spw=1.0):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
                         colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                         tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)

rows, balanced_sets = [], {}
for method in ["none", "class_weight", "under", "under+smote"]:
    Xb, yb = balance(method, X_tr, y_tr)
    balanced_sets[method] = (Xb, yb)
    spw = (yb == 0).sum() / max((yb == 1).sum(), 1) if method == "class_weight" else 1.0
    m = make_xgb(spw).fit(Xb, yb)
    prob = m.predict_proba(X_val)[:, 1]
    thr = best_threshold(y_val, prob)
    rows.append({"method": method, "train_fraud": int(yb.sum()), "train_rows": len(yb),
                 "threshold": round(thr, 3), **scores(y_val, prob, thr)})
    print("done:", method)

balance_table = pd.DataFrame(rows).sort_values("pr_auc", ascending=False).round(4)
balance_table.to_csv("reports/balancing_comparison_validation.csv", index=False)
balance_table

The best method is chosen automatically by **validation PR-AUC** (not accuracy, not test data).

In [ ]:
BEST_METHOD = balance_table.iloc[0]["method"]
print("Best balancing method:", BEST_METHOD)

X_bal, y_bal = balanced_sets[BEST_METHOD]
spw_best = (y_bal == 0).sum() / (y_bal == 1).sum() if BEST_METHOD == "class_weight" else 1.0

## 7. Compare 5 models (on the best balanced data)
The final model is chosen by **validation PR-AUC**. Test scores are only reported, so the test set never influences the choice.
Tip: `python train_model.py` runs the same pipeline from the terminal.

In [ ]:
RUN_KNN = True   # KNN is slow on large validation/test sets. Set False to skip.

spw_models = spw_best
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced" if BEST_METHOD == "class_weight" else None),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE,
                                            class_weight="balanced" if BEST_METHOD == "class_weight" else None),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=14, n_jobs=-1, random_state=RANDOM_STATE,
                                            class_weight="balanced" if BEST_METHOD == "class_weight" else None),
    "XGBoost": make_xgb(spw_models),
}
if RUN_KNN:
    models["KNN"] = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)

results, fitted, thresholds = [], {}, {}
for name, mdl in models.items():
    mdl.fit(X_bal, y_bal)
    val_prob = mdl.predict_proba(X_val)[:, 1]
    thr = best_threshold(y_val, val_prob)                  # threshold from validation
    test_prob = mdl.predict_proba(X_test)[:, 1]            # test is untouched / original, only reported
    results.append({"model": name, "threshold": round(thr, 3),
                    "val_pr_auc": average_precision_score(y_val, val_prob),   # used to CHOOSE the model
                    **scores(y_test, test_prob, thr)})
    fitted[name], thresholds[name] = mdl, thr
    print("done:", name)

comparison = pd.DataFrame(results).sort_values("val_pr_auc", ascending=False).round(4)
comparison.to_csv("reports/model_comparison_test.csv", index=False)
comparison

In [ ]:
ax = comparison.set_index("model")[["precision", "recall", "f1", "pr_auc"]].plot(kind="bar", figsize=(11, 5))
ax.set_title("Model comparison on the original test set")
ax.set_ylim(0, 1.05); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig("reports/model_comparison.png", dpi=150); plt.show()

## 8. Final model: confusion matrix, feature importance, save

In [ ]:
BEST_MODEL_NAME = comparison.iloc[0]["model"]
final_model, final_thr = fitted[BEST_MODEL_NAME], thresholds[BEST_MODEL_NAME]
print("Final model:", BEST_MODEL_NAME, "| balancing:", BEST_METHOD, "| threshold:", round(final_thr, 3))

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= final_thr).astype(int)
cm = confusion_matrix(y_test, test_pred)
print(cm)

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Genuine", "Fraud"], yticklabels=["Genuine", "Fraud"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix (test)")
plt.tight_layout(); plt.savefig("reports/confusion_matrix.png", dpi=150); plt.show()

if hasattr(final_model, "feature_importances_"):
    imp = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values()
    imp.plot(kind="barh", figsize=(7, 7), title="Feature importance")
    plt.tight_layout(); plt.savefig("reports/feature_importance.png", dpi=150); plt.show()

joblib.dump({"model": final_model, "threshold": final_thr, "scaler": scaler, "encoder": encoder,
             "features": FEATURES, "num_cols": num_cols, "cat_cols": cat_cols,
             "model_name": BEST_MODEL_NAME, "balancing": BEST_METHOD},
            "models/best_model.pkl")
print("Saved models/best_model.pkl")

## 9. SHAP - explainable AI
SHAP shows **which features pushed the score towards fraud (positive) or genuine (negative)**.  
Works for tree models (XGBoost, Random Forest, Decision Tree). If the final model is Logistic Regression or KNN, this section is skipped.

In [ ]:
# !pip install shap -q      # Colab / Jupyter: uncomment once
import shap

def fraud_values(sv):
    # shap returns different shapes for different models; always return (rows x features) for the FRAUD class
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1] if sv.shape[2] > 1 else sv[:, :, 0]
    return sv

try:
    explainer = shap.TreeExplainer(final_model)
    shap_sample = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_STATE)   # sample keeps it fast
    sv = fraud_values(explainer.shap_values(shap_sample))

    # (a) which features matter most overall + in which direction (red = high feature value)
    plt.figure()
    shap.summary_plot(sv, shap_sample, feature_names=FEATURES, show=False)
    plt.tight_layout(); plt.savefig("reports/shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

    # (b) simple bar chart of average importance
    plt.figure()
    shap.summary_plot(sv, shap_sample, feature_names=FEATURES, plot_type="bar", show=False)
    plt.tight_layout(); plt.savefig("reports/shap_bar.png", dpi=150, bbox_inches="tight"); plt.show()
except Exception as e:
    print("SHAP skipped:", e)

### Explain ONE fraud transaction that the model caught

In [ ]:
try:
    caught = np.where((y_test == 1) & (test_pred == 1))[0]
    if len(caught) == 0:
        print("No fraud transaction was caught in the test set.")
    else:
        i = caught[0]
        row_sv = fraud_values(explainer.shap_values(X_test.iloc[[i]]))[0]
        s = pd.Series(row_sv, index=FEATURES)
        top = s.reindex(s.abs().sort_values(ascending=False).index[:10])[::-1]
        raw_row = test_df.iloc[i]
        labels = [f"{f} = {raw_row[f]:.2f}" if isinstance(raw_row[f], (int, float, np.number)) else f"{f} = {raw_row[f]}"
                  for f in top.index]
        plt.figure(figsize=(8, 5))
        plt.barh(labels, top.values, color=["#E45756" if v > 0 else "#4C78A8" for v in top.values])
        plt.axvline(0, color="black", linewidth=0.8)
        plt.xlabel("Impact on fraud score (red = towards FRAUD, blue = towards GENUINE)")
        plt.title(f"Why test transaction #{i} was flagged (model score {test_prob[i]:.2f})")
        plt.tight_layout(); plt.savefig("reports/shap_single_fraud.png", dpi=150); plt.show()
except Exception as e:
    print("Single-transaction SHAP skipped:", e)